In [2]:
%matplotlib qt

import matplotlib.pyplot as plt

In [3]:
import sys
sys.path.insert(0, '.')

import numpy as np
import os

from config import NODES, EDGES, CALIBRATION_DATES, DATA_ROOT, calibration_path
from data_io import (
    load_session_df, load_sleap_dannce_keys, load_aligned_data,
    load_dannce_predictions, load_sleap_keys_3d,
)
from qc_utils import (
    find_sleap_dannce_alignment, compute_per_keypoint_distances,
    summarize_keypoint_distances, compute_session_qc_summary,
    track_distances_across_sessions, compute_temporal_alignment_quality,
    compute_keypoint_jitter, detect_tracking_dropouts,
    compute_bone_length_consistency,
    plot_per_keypoint_distances, plot_distance_over_time,
    plot_multi_session_distances, plot_temporal_alignment,
    generate_qc_video,
    test_calibration_epoch_effect, compute_days_since_calibration,
    plot_calibration_epoch_effect, plot_days_since_calibration,
    plot_snout_error_spatial_density,
)

import matplotlib.pyplot as plt
import scipy.io as sio

from sklearn.decomposition import PCA

from animations import animate_template

from config import NODES, EDGES, NODE_IDX
from data_io import (
    load_aligned_data, load_sleap_dannce_keys, load_template,
    load_behavior_log, load_session_df, load_frame_mapping, load_sleap_keys_3d,
    load_dannce_predictions,
)
from skeleton import normalize_skeleton_batch, project_to_pcs
from processing import get_template_match_indices, closest_indices

from scipy.ndimage import median_filter

In [4]:
rat = 'R1'
session = '2025_11_12_1'
datapath = "/home/yutaka-sprague/olveczky_lab/Lab/CLIRB/data"
folder = f'{rat}/{session}'

keys = load_sleap_dannce_keys(rat, session)
aligned = load_aligned_data(rat, session)

aligned_indices = aligned['dannce_idx_for_sleap_cams'].astype(int).ravel()[1:]

sleap_3d = keys['sleap_keys_3D']
sleap_3d = median_filter(keys['sleap_keys_3D'], size=(11, 1, 1))
dannce_3d = keys['dannce_keys_3D']
if dannce_3d.ndim == 4:
    dannce_3d = dannce_3d.squeeze(axis=1).transpose(0, 2, 1)
else:
    
    dannce_3d = np.transpose(dannce_3d, [0,2,1])

dannce_3d = dannce_3d[aligned_indices, :,:]

dannce_3d = median_filter(dannce_3d, size=(11, 1, 1))

print(f'SLEAP: {sleap_3d.shape}, DANNCE: {dannce_3d.shape}')

SLEAP: (36000, 23, 3), DANNCE: (36000, 23, 3)


## Check SLEAP/DANNCE alignment

In [5]:
# Find best alignment (Procrustes on 10 random frames, tries z-flip automatically)
alignment = find_sleap_dannce_alignment(
    sleap_3d, dannce_3d, n_sample_frames=10, seed=42, try_z_flip=True
)

print(f"Alignment residual: {alignment['residual']:.2f} calibration units")
print(f"Z-flipped: {alignment['z_flipped']}")
print(f"Scale factor: {alignment['s']:.4f}")
print(f"Rotation matrix:\n{alignment['R']}")
print(f"Translation: {alignment['t']}")

# Apply alignment and compute distances
sleap_aligned = alignment['apply'](sleap_3d)
distances = compute_per_keypoint_distances(sleap_aligned, dannce_3d)
summary = summarize_keypoint_distances(distances)

print(f"Overall mean distance: {summary['overall_mean']:.2f} calibration units")
print(f"Overall median distance: {summary['overall_median']:.2f} calibration units")
print(f"Valid frames: {summary['n_valid_frames']}")

Alignment residual: 9.61 calibration units
Z-flipped: True
Scale factor: 1.0051
Rotation matrix:
[[-0.99998093  0.00574112 -0.00227374]
 [-0.00575024 -0.99997537  0.00402484]
 [-0.00225057  0.00403784  0.99998932]]
Translation: [ 1.69300454 -2.39589975  0.73285078]
Overall mean distance: 10.09 calibration units
Overall median distance: 8.53 calibration units
Valid frames: 36000


In [6]:
# Distance over time (rolling mean)
fig, ax = plt.subplots(figsize=(14, 4))
plot_distance_over_time(
    distances, 
    keypoint_indices=[0, 3, 4, 10, 14, 18, 22],  # Snout, SpineF, SpineM, HandL, HandR, FootL, FootR
    window=200, ax=ax,
    title=f'{rat}/{session} — Keypoint distance over time'
)
plt.tight_layout()
plt.show()

In [7]:
output_video = f'qc_video_{rat}_{session}.mp4'

generate_qc_video(
    rat, session,
    output_path=output_video,
    alignment_result=alignment,
    n_frames=1000,
    start_frame=7200,
    fps=20,
    camera_idx=1,  # Camera1
    render_3d = True
)

Loading data for R2/2025_11_21_1...
Alignment residual: 65.73
Projecting SLEAP keypoints to 2D...
Transforming DANNCE to SLEAP space and projecting to 2D...
Rendering 1000 frames (with 3D panel)...


KeyboardInterrupt: 

In [7]:
from skeleton import project_to_pcs

# SLEAP: flip z, normalize, project — using TEMPLATE feature_means.
sleap_flipped = sleap_3d.copy()
sleap_flipped[:, :, 2] = -sleap_flipped[:, :, 2]
sleap_rotated, _, _ = normalize_skeleton_batch(sleap_flipped)

# DANNCE: normalize, project (no z-flip; DANNCE is already in template coords).
dannce_rotated, _, _ = normalize_skeleton_batch(dannce_3d)

sleap_dannce_stack = np.vstack((sleap_rotated, dannce_rotated))
sleap_dannce_flat = np.reshape(sleap_dannce_stack, (sleap_dannce_stack.shape[0], 69))

sleap_flat = np.reshape(sleap_rotated, (sleap_rotated.shape[0], 69))
dannce_flat = np.reshape(dannce_rotated, (dannce_rotated.shape[0], 69))

feat_means = np.mean(sleap_dannce_flat, axis=0)
feat_std = np.std(sleap_dannce_flat, axis=0)

pca = PCA()
pca.fit(sleap_dannce_flat)

pc_weights = pca.components_
pc_explained_variance = pca.explained_variance_ratio_

sleap_pcs = project_to_pcs(sleap_rotated, pc_weights, feat_means)
dannce_pcs = project_to_pcs(dannce_rotated, pc_weights, feat_means)

print(f'SLEAP PCs: {sleap_pcs.shape}')
print(f'DANNCE PCs: {dannce_pcs.shape}')

fig = plt.figure()

plt.plot(np.cumsum(pc_explained_variance))

plt.show()


SLEAP PCs: (36000, 69)
DANNCE PCs: (36000, 69)


In [39]:
print(pca.explained_variance_ratio_)

[7.06519908e-01 7.89215018e-02 4.96296958e-02 2.77926402e-02
 2.23856041e-02 2.10172384e-02 1.19900026e-02 9.49281637e-03
 7.68969591e-03 7.04896553e-03 6.51887427e-03 5.69329663e-03
 4.86605272e-03 3.31471302e-03 2.77425050e-03 2.64095365e-03
 2.43064601e-03 2.18959276e-03 1.97154710e-03 1.89364242e-03
 1.79416752e-03 1.37696155e-03 1.25619693e-03 1.09719132e-03
 1.08924500e-03 1.03111003e-03 9.07281304e-04 8.78105197e-04
 8.06883654e-04 7.82321677e-04 7.01962793e-04 6.72891641e-04
 6.66264467e-04 6.43272082e-04 6.16748051e-04 5.79250711e-04
 5.25572024e-04 5.03305343e-04 4.89943463e-04 4.56620993e-04
 4.37636127e-04 4.09276609e-04 4.02656862e-04 3.82795928e-04
 3.67157325e-04 3.51725577e-04 3.43821718e-04 3.33394637e-04
 3.25901272e-04 2.96497388e-04 2.66151648e-04 2.60258913e-04
 2.28555475e-04 2.23179436e-04 2.11594678e-04 2.00386225e-04
 1.89820752e-04 1.67285018e-04 1.61518044e-04 1.59768567e-04
 1.52449572e-04 1.40909532e-04 1.34673582e-04 1.13876762e-04
 8.17750090e-05 3.718869

In [8]:
# If sleap_keys_PCs exists in the data, compare
if 'sleap_keys_PCs' in keys:
    stored_pcs = keys['sleap_keys_PCs']
    fig, axes = plt.subplots(3, 1, figsize=(14, 6))
    n_show = min(2000, len(sleap_pcs))
    
    axes[0].plot(sleap_pcs[:n_show, 0], label='Computed PC1', alpha=0.7)
    axes[0].plot(dannce_pcs[:n_show,0], label='Computed dannce PC1', alpha=0.7)
    #axes[0].plot(stored_pcs[:n_show, 0], label='Stored PC1', alpha=0.7)
    axes[0].legend()
    axes[0].set_title('PC1 comparison')
    
    axes[1].plot(sleap_pcs[:n_show, 1], label='Computed PC2', alpha=0.7) 
    axes[1].plot(dannce_pcs[:n_show,1], label='Computed dannce PC2', alpha=0.7)
    #axes[1].plot(stored_pcs[:n_show, 1], label='Stored PC2', alpha=0.7)
    axes[1].legend()
    axes[1].set_title('PC2 comparison')

    axes[2].plot(sleap_pcs[:n_show, 2], label='Computed PC3', alpha=0.7)
    axes[2].plot(dannce_pcs[:n_show,2], label='Computed dannce PC3', alpha=0.7)
    #axes[1].plot(stored_pcs[:n_show, 1], label='Stored PC2', alpha=0.7)
    axes[2].legend()
    axes[2].set_title('PC3 comparison')
    
    plt.tight_layout()
    plt.show()

In [9]:
def get_keypoint_velocity(keys_3D_1, keys_3D_2):

    SpineF = keys_3D_1[3,:]
    SpineM = keys_3D_1[4,:]

    rotangle = np.arctan2( -(SpineF[1] - SpineM[1]), (SpineF[0] - SpineM[0]) )

    global_rotmat = np.zeros((2, 2))

    global_rotmat[0, 0] = np.cos(rotangle)
    global_rotmat[0, 1] = -np.sin(rotangle)
    global_rotmat[1, 0] = np.sin(rotangle)
    global_rotmat[1, 1] = np.cos(rotangle) 

    keys_1_centered = keys_3D_1 - SpineM 
    keys_2_centered = keys_3D_2 - SpineM 

    keys_1_rotated = keys_1_centered 
    keys_2_rotated = keys_2_centered 

    keys_1_rotated[:,:2] = np.transpose(global_rotmat @ np.transpose(keys_1_rotated[:,:2]))
    keys_2_rotated[:,:2] = np.transpose(global_rotmat @ np.transpose(keys_2_rotated[:,:2]))

    key_1_COM = np.mean(keys_1_rotated[3:5,:2], axis=0)
    key_2_COM = np.mean(keys_2_rotated[3:5,:2], axis=0)

    return key_2_COM-key_1_COM, global_rotmat # velocity centered on 

def get_COM_vel(keys_3D, sleap=False):
    COM_vel = np.zeros((keys_3D.shape[0], 2))

    for i in range(keys_3D.shape[0]):
        if i ==0:
            continue
        else:
            velocity, rotmat = get_keypoint_velocity(np.squeeze(keys_3D[i-1,:,:]), np.squeeze(keys_3D[i,:,:]))

            COM_vel[i,:] = velocity

    return COM_vel

dannce_COM_vel = get_COM_vel(dannce_3d)

## Fine tune SLEAP/DANNCE models and add corrections

In [10]:
# ---- Toggle ---------------------------------------------------------------
APPLY_CORRECTOR  = True
CORRECTOR_CKPT   = 'corrector/checkpoints/R1R2R3_world_temporal_mlp.pt'
CALIBRATION_MIN  = 5.0    # minutes of pre-task data used for the Procrustes fit
MAX_PROCRUSTES_RESIDUAL = 60.0   # mm; abort if SLEAP/DANNCE aren't co-calibrated
# ---------------------------------------------------------------------------

if APPLY_CORRECTOR:
    import sys, os
    # Ensure the corrector package is on the path. The notebook is run from
    # CLIRB_analyses/ in the existing setup, so a relative add works.
    sys.path.insert(0, os.path.abspath('.'))
    sys.path.insert(0, os.path.abspath('experiments'))
    import torch, numpy as np
    from corrector.world_alignment import calibration_indices, fit_procrustes
    from corrector.models import build_model

    # ---- Load the checkpoint ---------------------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(CORRECTOR_CKPT, map_location=device, weights_only=False)
    model_kwargs = dict(hidden=ck.get('hidden', 128),
                        n_hidden_layers=ck.get('n_hidden_layers', 2))
    if ck['model_name'] == 'temporal_mlp':
        model_kwargs['ctx'] = ck.get('ctx', 5)
    model = build_model(ck['model_name'], **model_kwargs)
    model.load_state_dict(ck['state_dict'])
    model = model.to(device).eval()
    eval_ctx = ck.get('ctx', 1)
    print(f'Loaded corrector: {ck["model_name"]}  ctx={eval_ctx}  '
          f'tag={ck.get("tag")}  best_val_mse={ck.get("best_val_mse"):.2f}')

    # ---- Procrustes fit on the calibration epoch -------------------------
    # SLEAP and DANNCE were already median-filtered above; for fitting we use
    # the same arrays that downstream uses (SLEAP world space, DANNCE world
    # space resampled to SLEAP frame rate via aligned_indices).
    sleap_world  = sleap_3d.astype(np.float32)         # (T, 23, 3)
    dannce_world = dannce_3d.astype(np.float32)        # already at SLEAP frame rate
    cal_idx = calibration_indices(len(sleap_world), CALIBRATION_MIN, sleap_hz=20.0,
                                  n_sample=1000, seed=0)
    tx = fit_procrustes(sleap_world[cal_idx], dannce_world[cal_idx], try_z_flip=True)
    print(f'Procrustes residual: {tx["residual"]:.2f} mm  scale={tx["s"]:.4f}  z_flipped={tx["z_flipped"]}')
    if tx['residual'] > MAX_PROCRUSTES_RESIDUAL:
        raise RuntimeError(
            f'Procrustes residual {tx["residual"]:.1f} mm exceeds threshold '
            f'{MAX_PROCRUSTES_RESIDUAL} mm — SLEAP and DANNCE may be uncalibrated. '
            f'Skip the corrector for this session.')

    # ---- Run the corrector ----------------------------------------------
    sl_aligned = tx['apply'](sleap_world).astype(np.float32)   # SLEAP -> DANNCE world space
    sl_corrected_dn_space = np.empty_like(sl_aligned)
    if eval_ctx <= 1:
        with torch.no_grad():
            for i in range(0, len(sl_aligned), 8192):
                xt = torch.from_numpy(sl_aligned[i:i+8192]).to(device)
                sl_corrected_dn_space[i:i+8192] = model(xt).cpu().numpy()
    else:
        T, ctx = len(sl_aligned), eval_ctx
        padded = np.concatenate([np.repeat(sl_aligned[:1], ctx-1, axis=0), sl_aligned], axis=0)
        with torch.no_grad():
            for i in range(0, T, 8192):
                idx = np.arange(i, min(i+8192, T))
                wins = np.stack([padded[s:s+ctx] for s in idx], axis=0)
                xt = torch.from_numpy(wins).to(device)
                sl_corrected_dn_space[i:i+len(idx)] = model(xt).cpu().numpy()
                
    sleap_corrected_world = tx['apply_inverse'](sl_corrected_dn_space).astype(np.float32)

    # ---- Replace sleap_3d so downstream uses corrected keypoints ---------
    sleap_3d_uncorrected = sleap_3d.copy()   # keep a copy in case you want to compare
    sleap_3d_corrected = sleap_corrected_world
    print(f'sleap_3d replaced with corrected keypoints. Original kept as sleap_3d_uncorrected.')
else:
    print('Corrector skipped (APPLY_CORRECTOR=False).')



Loaded corrector: temporal_mlp  ctx=5  tag=R1R2R3  best_val_mse=243.30
Procrustes residual: 10.76 mm  scale=1.0009  z_flipped=True
sleap_3d replaced with corrected keypoints. Original kept as sleap_3d_uncorrected.


In [17]:
sleap_flipped_corr = sleap_3d_corrected.copy()
sleap_flipped_corr[:, :, 2] = -sleap_flipped_corr[:,:,2]
sleap_rotated_corr, _, _ = normalize_skeleton_batch(sleap_flipped_corr)
sleap_flat_corr = np.reshape(sleap_rotated_corr, (sleap_rotated_corr.shape[0], 69))

sleap_pcs_corr = project_to_pcs(sleap_rotated_corr, pc_weights, feat_means)

fig, axes = plt.subplots(3, 1, figsize=(14, 6))
n_show = min(2000, len(sleap_pcs))

axes[0].plot(sleap_pcs[:n_show, 0], label='Computed SLEAP PC1', alpha=0.7)
axes[0].plot(dannce_pcs[:n_show,0], label='Computed DANNCE PC1', alpha=0.7)
axes[0].plot(sleap_pcs_corr[:n_show, 0], label='Corrected SLEAP PC1', alpha=0.7)
axes[0].legend()
axes[0].set_title('PC1 comparison')

axes[1].plot(sleap_pcs[:n_show, 1], label='Computed SLEAP PC2', alpha=0.7) 
axes[1].plot(dannce_pcs[:n_show,1], label='Computed DANNCE PC2', alpha=0.7)
axes[1].plot(sleap_pcs_corr[:n_show, 1], label='Corrected SLEAP PC3', alpha=0.7)
axes[1].legend()
axes[1].set_title('PC2 comparison')

axes[2].plot(sleap_pcs[:n_show, 2], label='Computed SLEAP PC3', alpha=0.7)
axes[2].plot(dannce_pcs[:n_show,2], label='Computed DANNCE PC3', alpha=0.7)
axes[2].plot(stored_pcs[:n_show, 2], label='Corrected SLEAP PC3', alpha=0.7)
axes[2].legend()
axes[2].set_title('PC3 comparison')

plt.tight_layout()
plt.show()

## Extract motifs from baseline sessions

Scan through session to identify motifs
Once template selected, search for matches in session
Set template as average of identified matches across baseline sessions
Save matched template

In [57]:
def sliding_windows(keypoints, win=30, step=10):
    # keypoints: (T, K, 3) or (K, 3, T) - we'll accept both
    kp = np.asarray(keypoints)
    if kp.ndim == 3 and kp.shape[0] == 3:  # maybe (3, K, T)
        kp = np.transpose(kp, (2,1,0))
    if kp.ndim == 3 and kp.shape[1] == 3 and kp.shape[2] != 3:
        # likely (T, K, 3)
        pass
    T, K, D = kp.shape
    windows = []
    idxs = []
    for start in range(0, T - win + 1, step):
        w = kp[start:start+win]  # shape (win, K, 3)
        windows.append(w)
        idxs.append(start)
    return np.array(windows), np.array(idxs)  # (n_windows, win, K, 3)

def mean_speed_of_window(window):
    # window: (win, K, 3)
    # compute speed per keypoint per frame, average
    diffs = np.diff(window, axis=0)  # (win-1, K, 3)
    speeds = np.linalg.norm(diffs, axis=2)  # (win-1, K)
    return speeds.mean()

def get_candidates(keys_3D, features, win_size=30, step=10, mov_thresh=1.0):

    windows, starts = sliding_windows(keys_3D, win=win_size, step=step)

    speeds = np.array([mean_speed_of_window(w) for w in windows])
    
    keep_mask = speeds >= mov_thresh
    kept_windows = windows[keep_mask]
    kept_starts = starts[keep_mask]
    print(f"Total windows: {len(windows)}, kept (moving): {len(kept_windows)}")

    return kept_starts

features_dannce = np.hstack((dannce_pcs, dannce_COM_vel))

candidates = get_candidates(dannce_3d, features_dannce, mov_thresh=2.5) #candidates should return frame indexes and template for each potential candidate, should also save each motif to a folder

Total windows: 3598, kept (moving): 1806


In [58]:
from collections import deque

def check_template_match(pc_buffer, pc_template, pc_template_bounds):
    """
    Check if PC values in buffer match the template within specified bounds.
    
    Parameters:
    - pc_buffer: Array of PC values, shape (buffer_length, n_components)
    - pc_template: Template PC trajectory, shape (template_length, n_components)
    - pc_template_bounds: Tolerance bounds for each PC dimension
    
    Returns:
    - True if buffer matches template within bounds, False otherwise
    """
    pc_array = np.array(pc_buffer)
    
    # Check how many points fall outside the bounds
    outside_bounds = (pc_array >= pc_template + pc_template_bounds) | (pc_array <= pc_template - pc_template_bounds)
    
    # If fewer than 3 points are outside bounds, consider it a match
    num_outside = np.sum(outside_bounds)
    
    return num_outside <= 3, num_outside


def get_template_match_indices(keys_pcs, pc_template, pc_template_bounds, refractory_frames=None):
    """
    Search through PC data to find frames matching a behavioral template.
    
    Parameters:
    - keys_pcs: Full PC trajectory data, shape (n_frames, n_components)
    - pc_template: Template PC trajectory to match, shape (template_length, n_components)
    - pc_template_bounds: Tolerance bounds for matching, same shape as pc_template
    - refractory_frames: Minimum frames between matches (default: template_length)
    
    Returns:
    - match_idxs: List of frame indices where template matches were found
    """
    buffer = deque()
    template_length = pc_template.shape[0]
    match_idxs = []
    num_outside = np.full(keys_pcs.shape[0], np.nan)
    
    # Set refractory period to template length if not specified
    if refractory_frames is None:
        refractory_frames = template_length
    
    last_match = -refractory_frames
        
    # Slide through all frames
    for i in range(keys_pcs.shape[0]):
        # Add current frame's PC values to buffer
        buffer.append(keys_pcs[i, :])
        
        # Remove old frames that exceed template length
        while buffer and len(buffer) > template_length:
            buffer.popleft()
        
        # Wait until buffer is full before checking
        if len(buffer) < template_length:
            continue
        
        # Check if buffer matches template AND refractory period has passed
        is_match, num_outside[i] = check_template_match(buffer, pc_template, pc_template_bounds)
        if is_match and (i - last_match >= refractory_frames):
            match_idxs.append(i)
            last_match = i
    
    return match_idxs, num_outside

In [59]:
def check_motifs(folder, keys_3D, features, candidates, pc_idxs, com_idxs, feature_means, feature_stds, bounds, win_size=30):

    save_folder = os.path.join(datapath, folder, 'motifs')
    
    pcs_use = features[:,pc_idxs]
    coms_use = features[:, com_idxs]
    stds_use = feature_stds[pc_idxs]

    good_motifs = {}

    prev_start = None 

    for start in candidates:

        if not prev_start:
            prev_start= start
        elif not start>prev_start +30:
            continue

        template = pcs_use[start:start+win_size]
        template_full = features[start:start+win_size]

        match_idxs, num_outside = get_template_match_indices(pcs_use, template, stds_use*bounds)

        if len(match_idxs)>5 and len(match_idxs)<50:
            prev_start = start
            print(f'Got {len(match_idxs)} matches for window starting at frame {start}')

            animate_template(keys_3D[start:start+30], template_full, feature_stds*bounds, feature_means, 20, 30, save_folder, f'frame_{start}_matches_{len(match_idxs)}', flip_skel=True)

            good_motifs[start] = match_idxs

    return good_motifs

In [60]:
good_motifs = check_motifs(folder, dannce_3d, features_dannce, candidates, [0,1], [2], feat_means, feat_std, 1.0, win_size=30)

Got 7 matches for window starting at frame 1870
Got 12 matches for window starting at frame 2020
Got 11 matches for window starting at frame 2350
Got 13 matches for window starting at frame 4620
Got 7 matches for window starting at frame 4860
Got 8 matches for window starting at frame 6220
Got 10 matches for window starting at frame 11660
Got 6 matches for window starting at frame 14050
Got 10 matches for window starting at frame 20330
Got 7 matches for window starting at frame 20480
Got 7 matches for window starting at frame 22720
Got 9 matches for window starting at frame 24450
Got 6 matches for window starting at frame 30410
Got 8 matches for window starting at frame 32000
Got 6 matches for window starting at frame 32780
Got 11 matches for window starting at frame 34890
Got 6 matches for window starting at frame 35070
Got 6 matches for window starting at frame 35670
Got 8 matches for window starting at frame 35720


In [ ]:
def save_checked_matches(idxs_to_check, features, check_folder, pcs_to_use, bounds, template_params):
    feature_stds = template_params['feature_stds']
    feature_means = template_params['feature_means']

    video_path = os.path.join(datapath, check_folder, 'sleap', 'camera1', '0.mp4')

    for temp_idx in idxs_to_check:

        output_vid_path = os.path.join(datapath, check_folder, 'sleap', f'template_matches_{temp_idx}.mp4')

        template_check = features[temp_idx:temp_idx+30,:]

        match_idxs, num_outside = get_template_match_indices(features_sleap[:,pcs_to_use], template_check[:,pcs_to_use], feature_stds[np.newaxis,[pcs_to_use]]*bounds, COM_dannce[:,0])

        print(match_idxs)

        save_template_matches_video(match_idxs, template_check, feature_stds[np.newaxis,:]*bounds, sleap_3D, features_sleap, edges, video_path, output_vid_path, max_matches=20)

## Evaluate overlap between SLEAP and DANNCE identified matches across motifs

In [ ]:
template_file = 'R1_template_1.npz'
n_components = 2  # number of PCs for template matching

# Load template
template_data = load_template(rat, template_file)
pc_template = template_data['template'][:,:n_components]  # (template_length, n_components)
pc_weights = template_data['pc_weights']  # (n_pcs, n_features)
feature_means = template_data['feature_means']
bounds = float(template_data['bounds'])
pcs_to_use = template_data['pcs_to_use']

template_filt = median_filter(pc_template, size=(5,1))

template_length = pc_template.shape[0]
print(f'Template length: {template_length} frames')
print(f'Bounds: {bounds}')
print(f'PCs used: {pcs_to_use}')

# Template bounds: uniform bounds applied per PC per timepoint      
feature_stds = template_data['feature_stds']
pc_template_bounds = np.ones_like(pc_template) * bounds * feature_stds[:n_components]

In [ ]:
# Run offline matching on SLEAP PCs
sleap_matches, num_outside = get_template_match_indices(
    sleap_pcs[:,:n_components], pc_template, pc_template_bounds, refractory_frames=template_length
    #sleap_pcs[:,:n_components], template_filt, pc_template_bounds, refractory_frames=template_length
    #sleap_pcs[:,:n_components], sleap_template_pcs[:,:2], pc_template_bounds, refractory_frames=template_length
)
print(f'SLEAP offline matches: {len(sleap_matches)}')

# Run offline matching on DANNCE PCs
dannce_matches, num_outside = get_template_match_indices(
    dannce_pcs[:,:n_components], pc_template, pc_template_bounds, refractory_frames=template_length
    #dannce_pcs[:,:n_components], template_filt, pc_template_bounds, refractory_frames=template_length
    #dannce_pcs[:,:n_components], dannce_template_pcs[:,:2], pc_template_bounds, refractory_frames=template_length
)
print(f'DANNCE offline matches: {len(dannce_matches)}')

In [ ]:
def plot_template_matches(pcs, match_indices, pc_template, pc_template_bounds,
                          n_components=2, title='', max_matches=20):
    """Plot matched PC trajectories against the template."""
    template_length = pc_template.shape[0]
    n_show = min(len(match_indices), max_matches)
    
    fig, axes = plt.subplots(1, n_components, figsize=(7 * n_components, 5))
    if n_components == 1: 
        axes = [axes]
    
    t_axis = np.arange(template_length)
    
    for pc in range(n_components):
        ax = axes[pc]
        # Template and bounds
        ax.plot(t_axis, pc_template[:, pc], 'k-', linewidth=2, label='Template')
        ax.fill_between(t_axis,
                        pc_template[:, pc] - pc_template_bounds[:, pc],
                        pc_template[:, pc] + pc_template_bounds[:, pc],
                        alpha=0.15, color='gray')
        
        # Matched trajectories
        for mi in match_indices[:n_show]:
            start = mi - template_length + 1
            if start < 0:
                continue
            segment = pcs[start:mi + 1, pc]
            ax.plot(t_axis[:len(segment)], segment, alpha=0.3, linewidth=1)
        
        ax.set_xlabel('Frame within template')
        ax.set_ylabel(f'PC{pc + 1}')
        ax.set_title(f'PC{pc + 1}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{title} — {n_show} matches shown', fontsize=12)
    plt.tight_layout()
    plt.show()


plot_template_matches(sleap_pcs, sleap_matches, template_filt, pc_template_bounds,
                      n_components=n_components, title='SLEAP offline matches')

plot_template_matches(dannce_pcs, dannce_matches, template_filt, pc_template_bounds,
                      n_components=n_components, title='DANNCE offline matches')

In [ ]:
# Plot match times over the session
fig, ax = plt.subplots(figsize=(14, 3))

if len(sleap_matches) > 0:
    ax.vlines(np.array(sleap_matches), 0.8, 1.2, colors='cyan', linewidth=1, label='SLEAP offline')

if len(dannce_matches) > 0:
    ax.vlines(np.array(dannce_matches), 1.8, 2.2, colors='magenta', linewidth=1, label='DANNCE offline')
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(['SLEAP', 'DANNCE'])
ax.set_xlabel('Frame') 
ax.set_title(f'{rat}/{session} — Template match timing')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
def closest_match(source_indices, target_indices):
    """
    For each t in target_times, return the index of the closest value in source_times.
    Returns empty array if source_times is empty.
    """

    return np.array([np.argmin(np.abs(np.array(source_indices)-t)) for t in target_indices])

def get_match_overlap(dannce_matches, sleap_matches, thresh=5):

    sleap_only = []
    dannce_only = []
    both = []

    if len(dannce_matches) == 0 and len(sleap_matches) == 0:
        return np.array(sleap_only), np.array(dannce_only), np.array(both)
    elif len(dannce_matches) == 0:
        return np.array(sleap_matches), np.array(dannce_only), np.array(both)
    elif len(sleap_matches) == 0:
        return np.array(sleap_only), np.array(dannce_matches), np.array(both)
    else:

        close_match_sleap = closest_match(dannce_matches, sleap_matches)
        close_match_dannce = closest_match(sleap_matches, dannce_matches)

        both = np.array(sleap_matches)[np.where(np.abs(np.array(sleap_matches) - np.array(dannce_matches)[close_match_sleap])<thresh)]
        sleap_only = np.array(sleap_matches)[np.where(np.abs(np.array(sleap_matches) - np.array(dannce_matches)[close_match_sleap])>=thresh)]
        dannce_only = np.array(dannce_matches)[np.where(np.abs(np.array(dannce_matches) - np.array(sleap_matches)[close_match_dannce])>=thresh)]
        
        return sleap_only, dannce_only, both

sleap_only, dannce_only, both = get_match_overlap(dannce_matches, sleap_matches)

In [ ]:
plot_template_matches(dannce_pcs, sleap_only, template_filt, pc_template_bounds,
                      n_components=n_components, title='SLEAP only matches using DANNCE PCs')

plot_template_matches(dannce_pcs, dannce_only, template_filt, pc_template_bounds,
                      n_components=n_components, title='DANNCE only matches using DANNCE PCs')

plot_template_matches(dannce_pcs, both, template_filt, pc_template_bounds,
                      n_components=n_components, title='Both matches using DANNCE PCs')


## Select starting template from motifs